# 01_dataset_prep.ipynb — End-to-End Colab GPU Fraud Training Bootstrap
This notebook covers dependency install, realistic synthetic data generation (200k-ready configurable), preprocessing, XGBoost GPU training, graph prep, and artifact export.

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm shap pyarrow fastparquet imbalanced-learn faker networkx torch

In [ ]:
import os, pandas as pd, numpy as np
from faker import Faker
fake = Faker('en_IN')

BASE='/content'
os.makedirs(f'{BASE}/data/synthetic', exist_ok=True)
os.makedirs(f'{BASE}/models', exist_ok=True)

try:
    import torch
    print('GPU:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except Exception as e:
    print(e)

## Realistic synthetic dataset generation

In [ ]:
N_ACCOUNTS=200
N_TXNS=50000   # increase to 200000 on Colab GPU runtime
FRAUD_RATIO=0.01

accounts=[f'ACC-{i:03d}' for i in range(1,N_ACCOUNTS+1)]
cities=['Mumbai','Delhi','Bengaluru','Pune','Hyderabad','Chennai','Kolkata','Ahmedabad']
fraud_types=['account_takeover','mule_ring','velocity_burst','impossible_geo','device_sharing','circular_laundering','new_beneficiary_spike']

rows=[]
for i in range(N_TXNS):
    sender=np.random.choice(accounts)
    receiver=np.random.choice([a for a in accounts if a!=sender])
    avg=np.random.lognormal(9,0.7)
    is_fraud=np.random.rand()<FRAUD_RATIO
    amount=float(np.random.lognormal(np.log(max(avg,1)),0.5))
    txns_last_2min=np.random.poisson(0.4)
    ring=np.random.beta(1,8)
    fraud_type=None
    if is_fraud:
        fraud_type=np.random.choice(fraud_types)
        amount*=np.random.uniform(2,6)
        txns_last_2min=np.random.randint(3,9)
        ring=np.random.uniform(0.7,0.98)
    rows.append({
        'txn_id': f'TX-{i:06d}',
        'timestamp': pd.Timestamp.now() - pd.Timedelta(minutes=np.random.randint(0,60*24*90)),
        'sender_id': sender,
        'receiver_id': receiver,
        'amount': round(amount,2),
        'location': np.random.choice(cities),
        'avg_30d_amount': round(avg,2),
        'device_change': int(np.random.rand()< (0.6 if is_fraud else 0.05)),
        'new_beneficiary_10min': int(np.random.rand()< (0.7 if is_fraud else 0.03)),
        'txns_last_2min': txns_last_2min,
        'geo_velocity': np.random.uniform(600,2500) if fraud_type=='impossible_geo' else np.random.exponential(30),
        'ring_risk_score': ring,
        'shared_device_count': np.random.randint(3,6) if fraud_type=='device_sharing' else 1,
        'fraud_label': int(is_fraud),
        'fraud_type': fraud_type
    })
df=pd.DataFrame(rows)
df.head(), df.fraud_label.mean()

In [ ]:
from sklearn.model_selection import train_test_split
df.to_parquet(f'{BASE}/data/synthetic/fraud_dataset.parquet', index=False)

train, temp = train_test_split(df, test_size=0.2, stratify=df['fraud_label'], random_state=42)
val, test = train_test_split(temp, test_size=0.5, stratify=temp['fraud_label'], random_state=42)

train.to_csv(f'{BASE}/data/synthetic/train.csv', index=False)
val.to_csv(f'{BASE}/data/synthetic/val.csv', index=False)
test.to_csv(f'{BASE}/data/synthetic/test.csv', index=False)

print(train.shape, val.shape, test.shape)

## Quick XGBoost GPU baseline

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score
import xgboost as xgb

work=train.copy()
for part in [work,val,test]:
    part['location']=LabelEncoder().fit_transform(part['location'])
    part['sender_enc']=part['sender_id'].str.extract(r'(\d+)').astype(int)
    part['receiver_enc']=part['receiver_id'].str.extract(r'(\d+)').astype(int)

features=['amount','avg_30d_amount','device_change','new_beneficiary_10min',
          'txns_last_2min','geo_velocity','ring_risk_score','shared_device_count',
          'location','sender_enc','receiver_enc']

X_train,y_train=work[features],work['fraud_label']
X_val,y_val=val[features],val['fraud_label']

model=xgb.XGBClassifier(
    n_estimators=200,max_depth=6,learning_rate=0.05,
    tree_method='hist',device='cuda'
)
model.fit(X_train,y_train)
pred=model.predict_proba(X_val)[:,1]
print('AUC-PR', average_precision_score(y_val,pred))
model.save_model(f'{BASE}/models/xgb_fraud_v1.json')

## Next steps
- Duplicate as `02_xgb_training.ipynb` for tuning
- Build `03_gnn_training.ipynb` from sender/receiver graph edges
- Export artifacts back to repo models/
- Use weekly retrain notebook for 7-day ModelOps